<a href="https://colab.research.google.com/github/Rundrayer/Evaluacion_Practica_Grupal_IDS221/blob/main/Calculo_Deficit1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:

# ==========================================
#IMPORTACIÓN Y RUTAS
# ==========================================

import pandas as pd
from pathlib import Path

# Importar las rutas definidas por el proyecto
from Configuracion import (
    DOC_ANALISIS,
    DOC_RESULTADOS,
    DT_DATASET,
    DT_PROCESADOS,
    DT_VISTA,
    DOC_REPORTERIA
)

# Rutas de los archivos de entrada
CSV_OPERACIONES = DT_DATASET / "operacion_sedes.csv"
CSV_SEDES = DT_DATASET / "sedes_geograficas.csv"

# Ruta del archivo de salida para ArcGIS
CSV_VALIDADO = DT_PROCESADOS / "geo_validado.csv"

# Mostrar las rutas utilizadas
print("IMPORTACIÓN Y RUTAS")
print("------------------------------")
print("Archivo de operaciones:", CSV_OPERACIONES)
print("Archivo de sedes:", CSV_SEDES)
print("Archivo de salida:", CSV_VALIDADO)


IMPORTACIÓN Y RUTAS
------------------------------
Archivo de operaciones: ../Datos/Datasets/operacion_sedes.csv
Archivo de sedes: ../Datos/Datasets/sedes_geograficas.csv
Archivo de salida: ../Datos/Procesados/geo_validado.csv


In [19]:
import sys
sys.path.insert(0, "/content")

from Configuracion import (
    DOC_ANALISIS,
    DOC_RESULTADOS,
    DT_DATASET,
    DT_PROCESADOS,
    DT_VISTA,
    DOC_REPORTERIA
)

print("Importación correcta")
print("Ruta de datasets:", DT_DATASET)
print("Ruta de procesados:", DT_PROCESADOS)

Importación correcta
Ruta de datasets: ../Datos/Datasets
Ruta de procesados: ../Datos/Procesados


In [11]:
from google.colab import files
archivos = files.upload()

Saving Configuracion.py to Configuracion.py


In [12]:
import os
import sys

sys.path.insert(0, "/content")

print("Archivo encontrado:", os.path.exists("/content/Configuracion.py"))

Archivo encontrado: True


In [13]:
from Configuracion import (
    DOC_ANALISIS,
    DOC_RESULTADOS,
    DT_DATASET,
    DT_PROCESADOS,
    DT_VISTA,
    DOC_REPORTERIA
)

print("Configuración importada correctamente")
print("Carpeta de datos:", DT_DATASET)
print("Carpeta de procesados:", DT_PROCESADOS)

Configuración importada correctamente
Carpeta de datos: ../Datos/Datasets
Carpeta de procesados: ../Datos/Procesados


In [14]:
from pathlib import Path

print("Carpeta actual:", Path.cwd())

print("¿Existe la carpeta de datos?", DT_DATASET.exists())
print("¿Existe la carpeta de procesados?", DT_PROCESADOS.exists())

print("\nArchivos en la carpeta de datos:")
if DT_DATASET.exists():
    print(list(DT_DATASET.iterdir()))
else:
    print("La carpeta no existe en esta ubicación.")

Carpeta actual: /content
¿Existe la carpeta de datos? True
¿Existe la carpeta de procesados? True

Archivos en la carpeta de datos:
[]


In [15]:
from pathlib import Path

base = Path("/content")

for archivo in base.rglob("*.csv"):
    print(archivo)

/content/sedes_geograficas.csv
/content/geo_validado.csv
/content/operacion_sedes.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv


In [16]:
from pathlib import Path
import shutil

# Crear las carpetas si todavía no existen
DT_DATASET.mkdir(parents=True, exist_ok=True)
DT_PROCESADOS.mkdir(parents=True, exist_ok=True)

# Copiar los archivos de entrada a la ruta configurada
for nombre in ["operacion_sedes.csv", "sedes_geograficas.csv"]:
    origen = Path("/content") / nombre
    destino = DT_DATASET / nombre

    if origen.exists() and origen.resolve() != destino.resolve():
        shutil.copy2(origen, destino)

# Guardar el archivo validado en la carpeta de procesados
validado = Path("/content/geo_validado.csv")
if validado.exists():
    shutil.copy2(validado, DT_PROCESADOS / "geo_validado.csv")

print("Archivos en Datasets:")
for archivo in DT_DATASET.iterdir():
    print(archivo)

print("\nArchivos en Procesados:")
for archivo in DT_PROCESADOS.iterdir():
    print(archivo)

Archivos en Datasets:
../Datos/Datasets/sedes_geograficas.csv
../Datos/Datasets/operacion_sedes.csv

Archivos en Procesados:
../Datos/Procesados/geo_validado.csv


In [17]:
import pandas as pd

CSV_OPERACIONES = DT_DATASET / "operacion_sedes.csv"
CSV_SEDES = DT_DATASET / "sedes_geograficas.csv"
CSV_VALIDADO = DT_PROCESADOS / "geo_validado.csv"

operaciones = pd.read_csv(CSV_OPERACIONES)
sedes = pd.read_csv(CSV_SEDES)
geo_validado = pd.read_csv(CSV_VALIDADO)

print("Operaciones:", operaciones.shape)
print("Sedes:", sedes.shape)
print("Datos geográficos validados:", geo_validado.shape)

print("\nColumnas de operaciones:")
print(operaciones.columns.tolist())

print("\nColumnas de sedes:")
print(sedes.columns.tolist())

Operaciones: (1620, 6)
Sedes: (9, 6)
Datos geográficos validados: (8, 4)

Columnas de operaciones:
['fecha', 'sede_id', 'region', 'generacion_kwh', 'demanda_kwh', 'evento_programado']

Columnas de sedes:
['sede_id', 'nombre_sede', 'municipio', 'region', 'latitud', 'longitud']


In [ ]:
#Punto 1
#Punto 1
import pandas as pd

operaciones = pd.read_csv("operacion_sedes.csv")
sedes = pd.read_csv("sedes_geograficas.csv")


operaciones["deficit_kwh"] = (
    operaciones["demanda_kwh"] - operaciones["generacion_kwh"]
).clip(lower=0)

#  Sumar el deficit acumulado de cada sede
deficit_sedes = operaciones.groupby(
    "sede_id", as_index=False
)["deficit_kwh"].sum()

#  Cambiar el nombre de la columna del deficit acumulado
deficit_sedes = deficit_sedes.rename(
    columns={"deficit_kwh": "deficit_total_kwh"}
)

# 5. Asociar el deficit con los datos geograficos
resultado = pd.merge(
    sedes,
    deficit_sedes,
    on="sede_id",
    how="left"
)

print("Deficit acumulado por sede:")
print(resultado.to_string(index=False))

#  Mostrar las sedes ordenadas de mayor a menor deficit
print("\nSedes ordenadas por deficit:")
print(
    resultado.sort_values(
        "deficit_total_kwh", ascending=False
    )[["sede_id", "nombre_sede", "deficit_total_kwh"]]
    .to_string(index=False)
)


Deficit acumulado por sede:
sede_id        nombre_sede    municipio    region  latitud  longitud  deficit_total_kwh
    S01   Solar Ahuachapán   Ahuachapán Occidente  13.9214  -89.8453           36692.27
    S02    Solar Santa Ana    Santa Ana Occidente  13.9942  -89.5597           20465.48
    S03    Solar Sonsonate    Sonsonate Occidente  13.7189  -89.7241             747.15
    S04 Solar San Salvador San Salvador    Centro  13.6929  -89.2182           77336.41
    S05  Solar Santa Tecla  Santa Tecla    Centro  13.6769  -89.2797           25528.84
    S06  Solar Cojutepeque  Cojutepeque    Centro  13.7179  -88.9331           10909.28
    S07   Solar San Miguel   San Miguel   Oriente  13.4837  -88.1779           69364.49
    S08     Solar Usulután     Usulután   Oriente  13.3500  -88.4461            3593.47
    S09     Solar La Unión     La Unión   Oriente  13.3369  -67.8440           42266.81

Sedes ordenadas por deficit:
sede_id        nombre_sede  deficit_total_kwh
    S04 Solar Sa

In [ ]:
#Punto 2
import pandas as pd

sedes = pd.read_csv("sedes_geograficas.csv")

# Convertir las coordenadas a valores numéricos
sedes["latitud"] = pd.to_numeric(
    sedes["latitud"], errors="coerce"
)

sedes["longitud"] = pd.to_numeric(
    sedes["longitud"], errors="coerce"
)

# Verificar la plausibilidad territorial
sedes["coordenadas_validas"] = (
    sedes["latitud"].between(13, 15)
    & sedes["longitud"].between(-91, -87)
)

# Separar las sedes válidas e inválidas
sedes_mapa = sedes[
    sedes["coordenadas_validas"]
].copy()

sedes_invalidas = sedes[
    ~sedes["coordenadas_validas"]
].copy()

print("SEDES VÁLIDAS PARA EL MAPA:")
print(sedes_mapa.to_string(index=False))

print("\nSEDES EXCLUIDAS DEL MAPA:")
print(sedes_invalidas.to_string(index=False))

print("\nRESUMEN:")
print("Total de sedes:", len(sedes))
print("Sedes válidas:", len(sedes_mapa))
print("Sedes excluidas:", len(sedes_invalidas))


SEDES VÁLIDAS PARA EL MAPA:
sede_id        nombre_sede    municipio    region  latitud  longitud  coordenadas_validas
    S01   Solar Ahuachapán   Ahuachapán Occidente  13.9214  -89.8453                 True
    S02    Solar Santa Ana    Santa Ana Occidente  13.9942  -89.5597                 True
    S03    Solar Sonsonate    Sonsonate Occidente  13.7189  -89.7241                 True
    S04 Solar San Salvador San Salvador    Centro  13.6929  -89.2182                 True
    S05  Solar Santa Tecla  Santa Tecla    Centro  13.6769  -89.2797                 True
    S06  Solar Cojutepeque  Cojutepeque    Centro  13.7179  -88.9331                 True
    S07   Solar San Miguel   San Miguel   Oriente  13.4837  -88.1779                 True
    S08     Solar Usulután     Usulután   Oriente  13.3500  -88.4461                 True

SEDES EXCLUIDAS DEL MAPA:
sede_id    nombre_sede municipio  region  latitud  longitud  coordenadas_validas
    S09 Solar La Unión  La Unión Oriente  13.3369   -6

In [ ]:

import pandas as pd
from google.colab import files

operaciones = pd.read_csv("operacion_sedes.csv")
sedes = pd.read_csv("sedes_geograficas.csv")

# 2. Calcular el déficit por registro
operaciones["deficit_kwh"] = (
    operaciones["demanda_kwh"] -
    operaciones["generacion_kwh"]
).clip(lower=0)

# 3. Calcular el déficit acumulado por sede
deficit_sedes = (
    operaciones.groupby("sede_id")["deficit_kwh"]
    .sum()
    .round(2)
    .reset_index(name="deficit_total_kwh")
)

# 4. Convertir las coordenadas a valores numéricos
sedes["latitud"] = pd.to_numeric(
    sedes["latitud"], errors="coerce"
)
sedes["longitud"] = pd.to_numeric(
    sedes["longitud"], errors="coerce"
)

# 5. Asociar los datos mediante sede_id
geo = sedes.merge(
    deficit_sedes,
    on="sede_id",
    how="left"
)

# 6. Validar las coordenadas
validas = (
    geo["latitud"].between(13, 15)
    & geo["longitud"].between(-91, -87)
    & geo["deficit_total_kwh"].notna()
)

# 7. Seleccionar las columnas solicitadas
geo_validado = geo.loc[
    validas,
    ["sede_id", "latitud", "longitud", "deficit_total_kwh"]
].copy()

# 8. Exportar el CSV con dos decimales
geo_validado.to_csv(
    "geo_validado.csv",
    index=False,
    float_format="%.2f"
)


# 10. Descargar el archivo
files.download("geo_validado.csv")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>